In [1]:
import requests
import pandas as pd
from datetime import datetime, date, timezone, timedelta
import time
import random
from typing import Any
import json
from pathlib import Path

from renewables_permitting.utils import as_list

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

# Funciones de parseo

In [ ]:
def parse_item(item: dict[str, Any], seccion: str, departamento: str, epigrafe: str | None, dt: datetime) -> dict[str, Any]:
    """
    Transforma una disposición o anuncio del BOE en un registro plano.

    Convierte la estructura jerárquica devuelta por la API de Sumarios
    del BOE en un diccionario con los campos normalizados que formarán
    parte del conjunto de datos final.

    Además de extraer los metadatos de la disposición, incorpora
    información contextual procedente de la sección, el departamento
    y la fecha de publicación.

    Parámetros
    ----------
    item : dict
        Nodo <item> de la API del BOE. Contiene la información de una
        disposición o anuncio publicado en el diario.

    seccion : str
        Nombre de la sección del BOE a la que pertenece la disposición.
        Ejemplo: "I. Disposiciones generales".

    departamento : str
        Nombre del departamento u organismo responsable de la publicación.
        Ejemplo: "MINISTERIO PARA LA TRANSICIÓN ECOLÓGICA Y EL RETO
        DEMOGRÁFICO".

    epigrafe : str | None
        Nombre del epígrafe al que pertenece el item. Será None cuando
        el item esté directamente dentro del nodo departamento.

    dt : datetime
        Fecha de publicación del BOE.

    Retorna
    -------
    dict
        Diccionario con la estructura normalizada del dataset:

        - identificador
        - titulo
        - url_html
        - pdf_link
        - seccion
        - departamento
        - epigrafe
        - site
        - place
        - date
        - year
        - month
        - day

    Notes
    -----
    El campo 'url_pdf' puede aparecer como una cadena o como un
    diccionario con metadatos adicionales (tamaño, páginas, etc.).
    En ambos casos se extrae únicamente la URL del documento PDF.

    Los campos 'site' y 'place' son valores constantes añadidos para
    facilitar la integración con otros conjuntos de datos.
    """
    url_pdf = item.get("url_pdf")

    return {
        "identificador": item.get("identificador"),
        "titulo": item.get("titulo"),
        "url_html": item.get("url_html"),
        "pdf_link": url_pdf.get("texto") if isinstance(url_pdf, dict) else url_pdf,
        "seccion": seccion,
        "departamento": departamento,
        "epigrafe": epigrafe,
        "site": "boe",
        "place": "espana",
        "date": dt.strftime("%Y/%m/%d"),
        "year": dt.year,
        "month": dt.month,
        "day": dt.day,
    }

In [ ]:
def parse_sumario_boe(data: dict[str, Any]) -> pd.DataFrame:
    """
    Convierte la respuesta JSON de la API de Sumarios del BOE en un DataFrame.

    Recorre la estructura jerárquica del sumario y extrae cada disposición
    o anuncio publicado junto con su contexto: sección, departamento y,
    cuando exista, epígrafe.

    La función contempla items ubicados dentro de ``epigrafe`` e items
    ubicados directamente dentro de ``departamento``.

    Parámetros
    ----------
    data : dict[str, Any]
        Respuesta JSON devuelta por la API de Sumarios del BOE.

    Retorna
    -------
    pd.DataFrame
        DataFrame con una fila por disposición o anuncio publicado.
    """
    rows: list[dict[str, Any]] = []

    try:
        sumario = data["data"]["sumario"]
    except KeyError as exc:
        raise ValueError(
            "La respuesta no contiene la estructura esperada: data.sumario."
        ) from exc

    fecha = sumario.get("metadatos", {}).get("fecha_publicacion")

    if not fecha:
        raise ValueError(
            "La respuesta no contiene 'fecha_publicacion' en los metadatos."
        )

    try:
        dt = datetime.strptime(fecha, "%Y%m%d")
    except ValueError as exc:
        raise ValueError(
            f"La fecha_publicacion no tiene formato AAAAMMDD válido: {fecha}"
        ) from exc

    for diario in as_list(sumario.get("diario")):
        for seccion in as_list(diario.get("seccion")):
            seccion_nombre = seccion.get("nombre")

            for departamento in as_list(seccion.get("departamento")):
                departamento_nombre = departamento.get("nombre")

                for epigrafe in as_list(departamento.get("epigrafe")):
                    epigrafe_nombre = epigrafe.get("nombre")

                    for item in as_list(epigrafe.get("item")):
                        rows.append(
                            parse_item(
                                item=item,
                                seccion=seccion_nombre,
                                departamento=departamento_nombre,
                                epigrafe=epigrafe_nombre,
                                dt=dt,
                            )
                        )

                for item in as_list(departamento.get("item")):
                    rows.append(
                        parse_item(
                            item=item,
                            seccion=seccion_nombre,
                            departamento=departamento_nombre,
                            epigrafe=None,
                            dt=dt,
                        )
                    )

    return pd.DataFrame(rows)

In [ ]:
df = parse_sumario_boe(data)
display(df.head(10))
display(df.loc[df["identificador"] == "BOE-A-2023-49"])
